# PubMed Biomedical Metadata — Exploratory Analysis

A tour of the dataset: scale, coverage over time, journals, MeSH topics, and how completeness of fields (keywords, conflict-of-interest statements) changes across the decades.

This is metadata only — no abstract text — so the focus is on the structure and shape of ~3 million biomedical article records (US-affiliated, human-subject, 1994–2025).

## Setup

## MOVE HERE IF IN NEED TO RUN CODE

- [2b. Trim the live edge](#2b-trim-the-live-edge-drop-2026)

In [ ]:
import os, glob, collections
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING — pick ONE option
# =====================================================================

# ---- OPTION A: LOCAL (active) -----------------------------------------
# Notebook runs from notebooks/, data sits one level up in the project root.
# Loading data with 2026 records for checking porposes
ROOT = os.path.dirname(os.getcwd())                       # go up one folder
DATA_DIR = os.path.join(ROOT, "data", "2_clean")          # full clean corpus (with abstracts)

# ---- OPTION B: KAGGLE (commented out — uncomment when running on Kaggle) ----
# # Data on Kaggle is filtered and has data up to 2025 already, so we can load the whole dataset for analysis.
# # The attached dataset lives under /kaggle/input/<dataset-slug>/.
# # This auto-detects the folder of Parquet shards so no path editing is needed.
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input — attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])

# =====================================================================
df = pd.read_parquet(DATA_DIR)
print(f"loaded {len(df):,} records from {DATA_DIR}")
print("columns:", list(df.columns))
df.head(3)

## 1. Scale and schema
The dataset is one row per article, keyed by PubMed ID (`uid`). Each record carries bibliographic metadata and flattened list fields (authors, affiliations, MeSH descriptors, keywords).

In [ ]:
print(f"records:       {len(df):,}")
print(f"unique PMIDs:  {df['uid'].nunique():,}")
print(f"year range:    {int(df['year'].min())}–{int(df['year'].max())}")
print(f"columns:       {df.shape[1]}")
df.dtypes

## 2. Publications per year
Annual volume grows steadily from the mid-1990s, accelerating through the 2010s. (The most recent year may be partial.)

In [ ]:
per_year = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.lineplot(x=per_year.index, y=per_year.values, marker="o", color="#1d6fb8")

# highlight the 2012–2015 anomaly window
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.15)          # shaded band
plt.axvline(2012, color="#e07a5f", ls="--", lw=1.2)            # left marker
plt.axvline(2015, color="#e07a5f", ls="--", lw=1.2)            # right marker

# overlay the dip segment in a different color
seg = per_year.loc[2012:2015]
sns.lineplot(x=seg.index, y=seg.values, marker="o", color="#e07a5f", lw=2.5)

plt.title("Articles per year (2012–2015 dip highlighted)")
plt.xlabel("year"); plt.ylabel("articles")
plt.tight_layout(); plt.show()

### The 2012–2015 dip

Article counts fall sharply from a 2012 peak to a 2014 low, then recover. This is very likely an **artifact of the affiliation filter**, not a real decline in output: PubMed's recording of author affiliations changed around 2013–2014 (all-author affiliations were captured more completely from ~2014 onward), and a query that filters on US affiliation under-matches records in the transition years. Actual biomedical publishing did not drop ~40% and rebound; the dip reflects how affiliation metadata was indexed, so counts in this window should be treated with caution.

## 2b. Trim the live edge (drop 2026)

The publications-per-year chart above shows a tiny tail in 2026: a handful of records carrying electronic / ahead-of-print dates, plus the still-accruing final months. These are the "live edge" of PubMed — not stable or reproducible, and a misleading sliver on any time series. For a clean, frozen 1994–2025 view, they are dropped here. This affects only this analysis; the underlying clean corpus is unchanged.

Already in export folder 

In [ ]:
import os, glob, collections
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = os.path.dirname(os.getcwd())                       # go up one folder
DATA_DIR = os.path.join(ROOT, "data", "2_clean")          # full clean corpus (with abstracts)

df = pd.read_parquet(os.path.join(ROOT, "data", "2_export", "abstracts_none"))

In [ ]:
per_year = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.lineplot(x=per_year.index, y=per_year.values, marker="o", color="#1d6fb8")

# highlight the 2012–2015 anomaly window
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.15)          # shaded band
plt.axvline(2012, color="#e07a5f", ls="--", lw=1.2)            # left marker
plt.axvline(2015, color="#e07a5f", ls="--", lw=1.2)            # right marker

# overlay the dip segment in a different color
seg = per_year.loc[2012:2015]
sns.lineplot(x=seg.index, y=seg.values, marker="o", color="#e07a5f", lw=2.5)

plt.title("Articles per year (2012–2015 dip highlighted)")
plt.xlabel("year"); plt.ylabel("articles")
plt.tight_layout(); plt.show()

## 3. Date precision
Most records carry imprecise publication dates (year-only or year+month). The `pubdate_precision` flag records this; month-level time analysis should use only `full_date` / `year_month` records.

In [ ]:
prec = df['pubdate_precision'].value_counts()
plt.figure(figsize=(7, 4))
sns.barplot(x=prec.index, y=prec.values, color="#1d6fb8")
plt.title("Publication-date precision"); plt.ylabel("records"); plt.xlabel("")
plt.tight_layout(); plt.show()
print((prec / len(df) * 100).round(1).astype(str) + " %")

## 4. Top journals
The corpus spans thousands of journals; a handful of high-volume titles dominate.

In [ ]:
print("distinct journals:", df['journal'].nunique())
top = df['journal'].value_counts().head(15)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
plt.title("Top 15 journals by article count"); plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

## 5. Authors per paper
Author counts are right-skewed: most papers have a handful of authors, with a long tail of large collaborations.

In [ ]:
print(df['n_authors'].describe().astype(int))
plt.figure(figsize=(10, 4))
sns.histplot(df['n_authors'].clip(upper=30), bins=30, color="#1d6fb8")
plt.title("Authors per paper (clipped at 30)"); plt.xlabel("authors"); plt.ylabel("papers")
plt.tight_layout(); plt.show()

## 6. Top MeSH topics
MeSH (Medical Subject Headings) are NLM's controlled vocabulary. The most frequent descriptors reflect the human-subject scope of the corpus.

In [ ]:
mc = collections.Counter(d for lst in df['mesh_descriptors'] for d in lst)
print("distinct MeSH descriptors:", len(mc))
top = pd.Series(dict(mc.most_common(15)))[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#2a9d5c")
plt.title("Top 15 MeSH descriptors"); plt.xlabel("occurrences"); plt.ylabel("")
plt.tight_layout(); plt.show()

In [ ]:
# Top 15 MeSH descriptors over time
TOP_N = 15
top_descriptors = [d for d, _ in mc.most_common(TOP_N)]

# explode mesh_descriptors so each descriptor gets its own row, then filter to top 15
mesh_exploded = (
    df[["year", "mesh_descriptors"]]
    .explode("mesh_descriptors")
    .rename(columns={"mesh_descriptors": "descriptor"})
)
mesh_exploded = mesh_exploded[mesh_exploded["descriptor"].isin(top_descriptors)]

# count per descriptor per year
mesh_year = (
    mesh_exploded
    .groupby(["year", "descriptor"])
    .size()
    .reset_index(name="count")
)

# plot
fig, ax = plt.subplots(figsize=(13, 6))
for desc in top_descriptors:
    sub = mesh_year[mesh_year["descriptor"] == desc]
    ax.plot(sub["year"], sub["count"], label=desc, linewidth=1.5)

ax.set_title("Top 15 MeSH descriptors — article count per year")
ax.set_xlabel("year")
ax.set_ylabel("articles")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# how many articles per year have zero MeSH vs at least one
mesh_coverage = df.groupby("year").agg(
    total=("uid", "count"),
    with_mesh=("n_mesh", lambda x: (x > 0).sum()),
    avg_mesh=("n_mesh", "mean")
).reset_index()
mesh_coverage["pct_with_mesh"] = mesh_coverage["with_mesh"] / mesh_coverage["total"] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mesh_coverage["year"], mesh_coverage["total"], label="total articles")
axes[0].set_title("total articles per year"); axes[0].set_xlabel("year")
axes[1].plot(mesh_coverage["year"], mesh_coverage["avg_mesh"], color="orange")
axes[1].set_title("avg MeSH descriptors per article per year"); axes[1].set_xlabel("year")
plt.tight_layout(); plt.show()

print(mesh_coverage[mesh_coverage["year"].between(2011, 2025)])

## 7. Field completeness over time
Three fields became standard in PubMed only gradually: keywords (from ~2012) and conflict-of-interest statements (from ~2017). Their coverage by year is a useful caveat for any longitudinal analysis.

In [ ]:
by_year = df.assign(
    has_kw=df['n_keywords'] > 0,
    has_mesh=df['n_mesh'] > 0,
).groupby('year').agg(
    keywords=('has_kw', 'mean'),
    coi=('has_coi', 'mean'),
    mesh=('has_mesh', 'mean'),
) * 100

plt.figure(figsize=(12, 4))
sns.lineplot(data=by_year, dashes=False, markers=False)
plt.title("Field coverage by year (%)"); plt.xlabel("year"); plt.ylabel("% of records")
plt.legend(title=""); plt.tight_layout(); plt.show()

MeSH coverage is 100% across all years — every record in this corpus carries at least one
descriptor. However, the average number of descriptors per article declined from ~13 (2010–2012)
to ~8 (2022–2023), reflecting NLM's transition to automated indexing. Analyses that rely on
MeSH breadth (e.g. topic co-occurrence) should account for this structural shift post-2012.

## 8. Working with the abstracts

In [ ]:
print("abstract text included:", (df['abstract'].fillna('').str.len() > 0).any())
print(f"mean original abstract length: {df['abstract_len'].mean():.0f} chars")
df['abstract_len'].clip(upper=4000).pipe(
    lambda s: sns.histplot(s, bins=50, color="#8a5fb0"))
plt.title("Original abstract length (chars, clipped 4000)"); plt.xlabel("chars")
plt.tight_layout(); plt.show()